In [1]:
import requests

base = "/lakehouse/default/Files/bronze/delinquances_ssmsi"
files = {
    "delinquance_communale.parquet": "https://www.data.gouv.fr/api/1/datasets/r/604d71b8-337d-4869-9226-49e01bae87df",
    "delinquance_departementale.csv": "https://www.data.gouv.fr/api/1/datasets/r/2b27a675-e3bf-41ef-a852-5fb9ab483967",
    "delinquance_regionale.csv": "https://www.data.gouv.fr/api/1/datasets/r/75753852-aca0-4c53-bfd4-a29513358d2f",
}
import os; os.makedirs(base, exist_ok=True)
for name, url in files.items():
    r = requests.get(url, allow_redirects=True, timeout=120)
    r.raise_for_status()
    with open(f"{base}/{name}", "wb") as f:
        f.write(r.content)

Silver

In [3]:
df = spark.read.parquet("Files/bronze/delinquances_ssmsi/delinquance_communale.parquet")
df.printSchema()

In [5]:
dep = spark.read.option("header", True).option("sep", ";").csv("Files/bronze/delinquances_ssmsi/delinquance_departementale.csv")
dep.printSchema()

In [6]:
df.select("indicateur").distinct().show(truncate=False)
df.select("est_diffuse").distinct().show()

Etape nettoyage

In [7]:
from pyspark.sql import functions as F

BRONZE = "Files/bronze/delinquances_ssmsi"

INDICATEURS = {
    "Vols de véhicule": "vol_vehicule",
    "Vols dans les véhicules": "vol_dans_vehicule",
    "Vols d'accessoires sur véhicules": "vol_accessoires_vehicule",
    "Destructions et dégradations volontaires": "degradation",
}
map_expr = F.create_map([F.lit(x) for kv in INDICATEURS.items() for x in kv])

def to_double(col):
    # gère "12,5", "NA", "" -> double ou null
    return F.regexp_replace(F.trim(F.col(col).cast("string")), ",", ".").cast("double")

# ===================== DÉPARTEMENTAL =====================
dep = (spark.read.option("header", True).option("sep", ";")
       .csv(f"{BRONZE}/delinquance_departementale.csv"))

dep_silver = (dep
    .withColumn("indicateur", F.trim("indicateur"))
    .filter(F.col("indicateur").isin(list(INDICATEURS)))
    .select(
        F.upper(F.lpad(F.trim("Code_departement"), 2, "0")).alias("code_departement"),
        F.lpad(F.trim("Code_region"), 2, "0").alias("code_region"),
        F.col("annee").cast("int").alias("annee"),
        map_expr[F.col("indicateur")].alias("indicateur_code"),
        F.col("indicateur"),
        F.col("unite_de_compte"),
        to_double("nombre").cast("long").alias("nombre"),
        to_double("taux_pour_mille").alias("taux_pour_mille"),
        to_double("insee_pop").cast("long").alias("population"),
        F.col("insee_pop_millesime").cast("int").alias("population_millesime"),
    )
    .withColumn("_source", F.lit("SSMSI - base départementale"))
    .withColumn("_ingested_at", F.current_timestamp())
    .dropDuplicates(["code_departement", "annee", "indicateur_code"]))

dep_silver.write.mode("overwrite").format("delta") \
    .option("overwriteSchema", "true").saveAsTable("silver_delinquance_departement")

# ===================== COMMUNAL =====================
com = spark.read.parquet(f"{BRONZE}/delinquance_communale.parquet")

com_silver = (com
    .withColumn("indicateur", F.trim("indicateur"))
    .filter(F.col("indicateur").isin(list(INDICATEURS)))
    .withColumn("code_commune", F.upper(F.lpad(F.trim("CODGEO_2026"), 5, "0")))
    .withColumn("est_diffuse_bool", F.col("est_diffuse") == "diff")
    .select(
        "code_commune",
        F.when(F.col("code_commune").startswith("97"), F.substring("code_commune", 1, 3))
         .otherwise(F.substring("code_commune", 1, 2)).alias("code_departement"),
        "annee",
        map_expr[F.col("indicateur")].alias("indicateur_code"),
        "indicateur",
        "unite_de_compte",
        # secret statistique : null si non diffusé, jamais 0 inventé
        F.when(F.col("est_diffuse_bool"), F.col("nombre")).alias("nombre"),
        F.when(F.col("est_diffuse_bool"), F.col("taux_pour_mille")).alias("taux_pour_mille"),
        F.col("est_diffuse_bool").alias("est_diffuse"),
        "complement_info_nombre",
        "complement_info_taux",
        F.col("insee_pop").alias("population"),
        F.col("insee_pop_millesime").alias("population_millesime"),
    )
    .withColumn("_source", F.lit("SSMSI - base communale"))
    .withColumn("_ingested_at", F.current_timestamp())
    .dropDuplicates(["code_commune", "annee", "indicateur_code"]))

com_silver.write.mode("overwrite").format("delta") \
    .option("overwriteSchema", "true") \
    .partitionBy("annee").saveAsTable("silver_delinquance_commune")